# Causal Extraction Performance Analysis

**Purpose:** Show-and-tell evaluation of the causal extraction pipeline against the annotated dataset in `data/causal_dataset.json`.

| Section | What it tests |
|---------|---------------|
| 1 — Extraction | Run `extract_stage5_causal_condition` on every entry; display per-entry results |
| 2 — Span quality | Token-F1 for cause/effect spans vs ground truth (Stage 1: noun-chunk spans) |
| 3 — Connector direction | Verify forward/backward connectors are correctly oriented (direction-fix) |
| 4 — Chain detection | Node-overlap and reconstruction accuracy for multi-hop entries (Stage 2) |
| 5 — Entity linking | `_best_overlapping_entity` hit-rate vs `expected_g4_mechanism` (Stage 3) |
| 6 — Threshold calibration | Jaccard sweep to find optimal `_CHAIN_JACCARD_THRESHOLD` and `_ENTITY_LINK_JACCARD_THRESHOLD` |
| 7 — Summary dashboard | Aggregate metrics across all stages |

In [ ]:
import sys, os, json, re
import pandas as pd
import numpy as np

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
SRC_DIR      = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..', '..'))
NER_PKG_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', 'ner'))
DATA_DIR     = os.path.join(NOTEBOOK_DIR, 'data')

for p in [SRC_DIR, NER_PKG_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import spacy
nlp = spacy.load('en_core_web_sm')

from dackar.RCA.ner.causal_condition_adapter import (
    extract_stage5_causal_condition,
    _chain_causal_statements,
    _CHAIN_JACCARD_THRESHOLD,
)
from dackar.RCA.doc_extraction.adapter import (
    _best_overlapping_entity,
    _ENTITY_LINK_JACCARD_THRESHOLD,
)
from dackar.RCA.ner.hybrid_ner.models import ResolvedSpan

print('Setup OK')
print(f'  SRC_DIR                       : {SRC_DIR}')
print(f'  _CHAIN_JACCARD_THRESHOLD      : {_CHAIN_JACCARD_THRESHOLD}')
print(f'  _ENTITY_LINK_JACCARD_THRESHOLD: {_ENTITY_LINK_JACCARD_THRESHOLD}')

# J3 — word vector check
_has_vectors = nlp.vocab.vectors.shape[0] > 0
if not _has_vectors:
    print()
    print('WARNING: no word vectors in this spaCy model (en_core_web_sm).')
    print('         Improvement D (embedding chain linking) is disabled.')
    print('         Re-run with en_core_web_md or en_core_web_lg to evaluate that path.')
else:
    print(f'  Word vectors                  : {nlp.vocab.vectors.shape[0]} ({nlp.vocab.vectors.shape[1]}-dim)')


---
## Section 0 — CausalSentence / CausalSimple smoke test (14.5)

Regression gate for the **primary extractor paths** (`CausalSentence`, `CausalSimple`).
These paths require SSC entity annotations from the upstream NER pipeline and are
invisible in the main evaluation (which runs on raw text only).

Each fixture injects mock SSC + causal-keyword entity patterns via
`causal_sentence_factory` and verifies that:
1. `CausalSentence` fires (not dep_fallback)
2. `extractor_used == "CausalSentence"`
3. At least one statement is extracted with non-empty cause and effect
4. `source` field in extracted statements is NOT `"dep_fallback"`

Run this cell before any merge of Improvements G–J to catch regressions.

In [ ]:
import sys as _sys
import dackar.config as _dackar_cfg
import dackar.utils  as _dackar_utils
_sys.modules.setdefault('config', _dackar_cfg)
_sys.modules.setdefault('utils',  _dackar_utils)
import dackar.text_processing as _dackar_tp
_sys.modules.setdefault('text_processing', _dackar_tp)

from dackar.RCA.causal.CausalSentence import CausalSentence

# ── Fixture builder ───────────────────────────────────────────────────────────
def _make_cs_factory(ssc_cause, ssc_effect, causal_connector):
    """Return a causal_sentence_factory that pre-injects SSC + causal entities."""
    def factory(text, nlp):
        cs = CausalSentence(nlp)
        cs.addEntityPattern('SSC_pattern', [
            {'label': 'SSC',    'pattern': ssc_cause,         'id': 'SSC'},
            {'label': 'SSC',    'pattern': ssc_effect,        'id': 'SSC'},
        ])
        cs.addEntityPattern('causal_pattern', [
            {'label': 'causal', 'pattern': causal_connector,  'id': 'causal'},
        ])
        cs(text)
        return cs
    return factory

# ── Smoke test fixtures ───────────────────────────────────────────────────────
_SMOKE_FIXTURES = [
    {
        'id':        'smoke-01',
        'text':      'The bearing caused seal failure.',
        'ssc_cause': 'bearing', 'ssc_effect': 'seal failure',
        'connector': 'caused',
        'doc_type':  'CR',
    },
    {
        'id':        'smoke-02',
        'text':      'Pump cavitation caused impeller erosion.',
        'ssc_cause': 'Pump cavitation', 'ssc_effect': 'impeller erosion',
        'connector': 'caused',
        'doc_type':  'WO',
    },
    {
        'id':        'smoke-03',
        'text':      'Coolant leakage led to core damage.',
        'ssc_cause': 'Coolant leakage', 'ssc_effect': 'core damage',
        'connector': 'led',
        'doc_type':  'CR',
    },
    {
        'id':        'smoke-04',
        'text':      'Valve stem corrosion resulted in loss of isolation.',
        'ssc_cause': 'Valve stem corrosion', 'ssc_effect': 'loss of isolation',
        'connector': 'resulted',
        'doc_type':  'CR',
    },
    {
        'id':        'smoke-05',
        'text':      'Rotor imbalance caused high shaft vibration.',
        'ssc_cause': 'Rotor imbalance', 'ssc_effect': 'high shaft vibration',
        'connector': 'caused',
        'doc_type':  'WO',
    },
]

# ── Run and assert ────────────────────────────────────────────────────────────
BAR = '─' * 70
smoke_results = []
n_pass = n_fail = 0

for fix in _SMOKE_FIXTURES:
    factory = _make_cs_factory(fix['ssc_cause'], fix['ssc_effect'], fix['connector'])
    result  = extract_stage5_causal_condition(
        doc_id=fix['id'], chunk_index=0, chunk_text=fix['text'],
        doc_type=fix['doc_type'], section_role='body', nlp=nlp,
        causal_sentence_factory=factory,
    )
    stmts       = result.get('extracted_causal_statements', [])
    ext_used    = result['extractor']['used']
    has_stmt    = bool(stmts)
    cs_fired    = ext_used == 'CausalSentence'
    cause_ok    = any(s.get('cause_text') for s in stmts)
    effect_ok   = any(s.get('effect_text') for s in stmts)
    no_fallback = all(s.get('source', '') != 'dep_fallback' for s in stmts)

    passed = cs_fired and has_stmt and cause_ok and effect_ok and no_fallback
    n_pass += passed; n_fail += not passed

    smoke_results.append({
        'id': fix['id'], 'pass': passed,
        'extractor_used': ext_used,
        'n_stmts': len(stmts),
        'cause': stmts[0].get('cause_text','') if stmts else '',
        'effect': stmts[0].get('effect_text','') if stmts else '',
        'source': stmts[0].get('source','') if stmts else '',
        'checks': {
            'cs_fired': cs_fired, 'has_stmt': has_stmt,
            'cause_ok': cause_ok, 'effect_ok': effect_ok,
            'no_fallback': no_fallback,
        },
    })

print(BAR)
print(f'CausalSentence smoke test: {n_pass}/{len(_SMOKE_FIXTURES)} PASS'
      + ('' if n_fail == 0 else f'  *** {n_fail} FAIL ***'))
print(BAR)
for r in smoke_results:
    status = '✓ PASS' if r['pass'] else '✗ FAIL'
    print(f"[{r['id']}] {status}  extractor={r['extractor_used']}  "
          f"stmts={r['n_stmts']}")
    if not r['pass']:
        print(f"        checks: {r['checks']}")
    else:
        print(f"        cause={r['cause']!r}  effect={r['effect']!r}  "
              f"source={r['source']!r}")
print(BAR)
if n_fail:
    print('ACTION REQUIRED: CausalSentence path is broken — do not merge G–J changes.')
else:
    print('Primary extractor path is healthy. Safe to proceed with G–J implementation.')


## Load dataset

In [ ]:

# ── Span offset helper (Fix 5) ────────────────────────────────────────────────
def _resolve_span_offset(text: str, span: str):
    idx = text.find(span)
    if idx == -1:
        return -1, -1
    return idx, idx + len(span)

# ── Chain builder ─────────────────────────────────────────────────────────────
def _build_chain_from_rels(rels):
    if not rels:
        return None
    causes  = {r['cause_span'] for r in rels}
    effects = {r['effect_span'] for r in rels}
    adj = {}
    for r in rels:
        adj.setdefault(r['cause_span'], []).append(r['effect_span'])
    roots = [c for c in causes if c not in effects] or list(causes)[:1]
    best = []
    def dfs(node, path):
        nonlocal best
        nexts = [n for n in adj.get(node, []) if n not in path]
        if not nexts:
            if len(path) > len(best): best = list(path)
            return
        for n in nexts: dfs(n, path + [n])
    for root in roots:
        dfs(root, [root])
    return best if len(best) >= 2 else None

# ── Text-list normalizer ───────────────────────────────────────────────────────
def _join_text_list(items):
    """Join a list that may be plain strings or {paragraph_index, text} dicts."""
    parts = []
    for item in (items or []):
        if isinstance(item, str):
            parts.append(item)
        elif isinstance(item, dict):
            parts.append(str(item.get('text', '')))
    return ' '.join(parts)

# ── Dataset loaders ───────────────────────────────────────────────────────────
def _load_ds1(entries):
    out = []
    for e in entries:
        out.append({
            'id': e['id'], 'source': e['source'], 'dataset': 'ds1',
            'text': e['text'],
            'relations': [{'cause_span': e['cause']['span'],
                           'effect_span': e['effect']['span'],
                           'cause_start': e['cause'].get('start', -1),
                           'cause_end':   e['cause'].get('end',   -1),
                           'effect_start': e['effect'].get('start', -1),
                           'effect_end':   e['effect'].get('end',   -1),
                           'connective': e.get('causal_connective', ''),
                           'relation_type': e.get('relation_type', '')}],
            'chain': e.get('chain'),
            'connector_direction': e.get('connector_direction'),
            'expected_g4_mechanism': e.get('expected_g4_mechanism'),
            'challenge_type': '',
            'causal_status': '',
            'attribution': '',
        })
    return out

def _load_ds2(entries):
    out = []
    for e in entries:
        rels = [{'cause_span': e['cause']['span'],
                 'effect_span': e['effect']['span'],
                 'cause_start': e['cause'].get('start', -1),
                 'cause_end':   e['cause'].get('end',   -1),
                 'effect_start': e['effect'].get('start', -1),
                 'effect_end':   e['effect'].get('end',   -1),
                 'connective': e.get('causal_connective', ''),
                 'relation_type': e.get('relation_type', '')}]
        chain = None
        if e.get('secondary_effect'):
            rels.append({'cause_span': e['effect']['span'],
                         'effect_span': e['secondary_effect']['span'],
                         'cause_start': e['effect'].get('start', -1),
                         'cause_end':   e['effect'].get('end',   -1),
                         'effect_start': e['secondary_effect'].get('start', -1),
                         'effect_end':   e['secondary_effect'].get('end',   -1),
                         'connective': '', 'relation_type': 'explicit'})
            chain = [e['cause']['span'], e['effect']['span'],
                     e['secondary_effect']['span']]
        out.append({
            'id': e['id'], 'source': e['source'], 'dataset': 'ds2',
            'text': e['text'],
            'relations': rels, 'chain': chain,
            'connector_direction': None, 'expected_g4_mechanism': None,
            'challenge_type': '', 'causal_status': '', 'attribution': '',
        })
    return out

def _load_ds3(entries):
    out = []
    unresolved = []
    for e in entries:
        text = e['text']
        rels = []
        for r in e.get('causal_relations', []):
            cspan = r['cause']['span']
            espan = r['effect']['span']
            cs, ce = _resolve_span_offset(text, cspan)
            es, ee = _resolve_span_offset(text, espan)
            if cs == -1: unresolved.append((e['id'], 'cause', cspan))
            if es == -1: unresolved.append((e['id'], 'effect', espan))
            rels.append({
                'cause_span':   cspan,  'effect_span':  espan,
                'cause_start':  cs,     'cause_end':    ce,
                'effect_start': es,     'effect_end':   ee,
                'connective':   r.get('causal_connective', ''),
                'relation_type': r.get('relation_type', ''),
            })
        out.append({
            'id': e['id'], 'source': e['source'], 'dataset': 'ds3',
            'text': text, 'relations': rels,
            'chain': _build_chain_from_rels(rels),
            'connector_direction': None, 'expected_g4_mechanism': None,
            'challenge_type': '', 'causal_status': '', 'attribution': '',
        })
    if unresolved:
        print(f'  DS3 unresolved spans ({len(unresolved)}): {unresolved}')
    return out

def _load_ds4(entries):
    out = []
    unresolved = []
    for e in entries:
        text = ' '.join(e['sentences'])
        rels = []
        for r in e.get('cross_sentence_causal_relations', []):
            cspan = r['cause']['span']
            espan = r['effect']['span']
            cs, ce = _resolve_span_offset(text, cspan)
            es, ee = _resolve_span_offset(text, espan)
            if cs == -1: unresolved.append((e['id'], 'cause', cspan))
            if es == -1: unresolved.append((e['id'], 'effect', espan))
            rels.append({
                'cause_span':   cspan,  'effect_span':  espan,
                'cause_start':  cs,     'cause_end':    ce,
                'effect_start': es,     'effect_end':   ee,
                'connective':   r.get('causal_connective', ''),
                'relation_type': r.get('relation_type', ''),
            })
        out.append({
            'id': e['id'], 'source': e['source'], 'dataset': 'ds4',
            'text': text, 'relations': rels,
            'chain': _build_chain_from_rels(rels),
            'connector_direction': None, 'expected_g4_mechanism': None,
            'challenge_type': '', 'causal_status': '', 'attribution': '',
        })
    if unresolved:
        print(f'  DS4 unresolved spans ({len(unresolved)}): {unresolved}')
    return out

def _load_ds5(entries):
    """Cross-sentence / cross-paragraph challenge dataset (30 records).
    12 challenge types. XPG-* entries use 'paragraphs' (list of dicts with 'text')
    instead of 'sentences' (list of strings) — _join_text_list() handles both.
    Multi-cause records use cause_set; expanded to individual (cause_i, effect) pairs.
    """
    out = []
    unresolved = []
    for e in entries:
        # XPG-001/XPG-002: paragraphs is a list of {paragraph_index, text} dicts
        text = _join_text_list(e.get('sentences') or e.get('paragraphs') or [])
        relations_raw = (
            e.get('cross_sentence_causal_relations')
            or e.get('cross_paragraph_causal_relations')
            or []
        )
        rels = []
        for r in relations_raw:
            if 'cause_set' in r:
                espan = r['effect']['span']
                es, ee = _resolve_span_offset(text, espan)
                if es == -1: unresolved.append((e['id'], 'effect', espan))
                for c_entry in r['cause_set']:
                    cspan = c_entry['span']
                    cs, ce = _resolve_span_offset(text, cspan)
                    if cs == -1: unresolved.append((e['id'], 'cause', cspan))
                    rels.append({
                        'cause_span': cspan, 'effect_span': espan,
                        'cause_start': cs, 'cause_end': ce,
                        'effect_start': es, 'effect_end': ee,
                        'connective': r.get('causal_connective', ''),
                        'relation_type': r.get('relation_type', 'explicit'),
                    })
            elif 'cause' in r and 'effect' in r:
                cspan = r['cause']['span']
                espan = r['effect']['span']
                cs, ce = _resolve_span_offset(text, cspan)
                es, ee = _resolve_span_offset(text, espan)
                if cs == -1: unresolved.append((e['id'], 'cause', cspan))
                if es == -1: unresolved.append((e['id'], 'effect', espan))
                rels.append({
                    'cause_span': cspan, 'effect_span': espan,
                    'cause_start': cs, 'cause_end': ce,
                    'effect_start': es, 'effect_end': ee,
                    'connective': r.get('causal_connective', ''),
                    'relation_type': r.get('relation_type', ''),
                })
        out.append({
            'id': e['id'], 'source': e.get('source', 'condition_report'),
            'dataset': 'ds5', 'text': text, 'relations': rels,
            'chain': _build_chain_from_rels(rels),
            'connector_direction': None, 'expected_g4_mechanism': None,
            'challenge_type': e.get('challenge_type', ''),
            'causal_status': '', 'attribution': '',
        })
    if unresolved:
        print(f'  DS5 unresolved spans ({len(unresolved)}): {unresolved}')
    return out

def _load_ds6a(entries):
    """Advanced challenge dataset (44 records, 97 relations, 8 challenge types, 2 tiers)."""
    out = []
    unresolved = []
    for e in entries:
        text = ' '.join(e.get('sentences', [e.get('text', '')]))
        rels = []
        for r in e.get('causal_relations', []):
            cause = r.get('cause', {})
            effect = r.get('effect', {})
            cspan = cause.get('span', '') if isinstance(cause, dict) else str(cause)
            espan = effect.get('span', '') if isinstance(effect, dict) else str(effect)
            if not cspan or not espan:
                continue
            cs, ce = _resolve_span_offset(text, cspan)
            es, ee = _resolve_span_offset(text, espan)
            if cs == -1: unresolved.append((e['id'], 'cause', cspan))
            if es == -1: unresolved.append((e['id'], 'effect', espan))
            rels.append({
                'cause_span':    cspan,  'effect_span':   espan,
                'cause_start':   cs,     'cause_end':     ce,
                'effect_start':  es,     'effect_end':    ee,
                'connective':    r.get('causal_connective', ''),
                'relation_type': r.get('relation_type', ''),
                'causal_status': r.get('causal_status', ''),
                'attribution':   r.get('attribution', ''),
                'embedded_in':   r.get('embedded_in', ''),
            })
        out.append({
            'id': e['id'], 'source': e.get('source', 'condition_report'),
            'dataset': 'ds6a', 'text': text, 'relations': rels,
            'chain': _build_chain_from_rels(rels),
            'connector_direction': None, 'expected_g4_mechanism': None,
            'challenge_type': e.get('challenge_type', ''),
            'tier': e.get('tier', 1),
            'causal_status': '',
            'attribution':   '',
        })
    if unresolved:
        print(f'  DS6a unresolved spans ({len(unresolved)}): {unresolved}')
    return out

# ── Load ─────────────────────────────────────────────────────────────────────
all_entries = []
_DS1_LOADED = False
for _fname in ('causal_dataset.json', 'causal_dataset_1.json'):
    _fpath = os.path.join(DATA_DIR, _fname)
    if os.path.exists(_fpath):
        raw = json.load(open(_fpath))
        all_entries.extend(_load_ds1(raw))
        print(f'Loaded ds1 from {_fname}: {len(raw)} entries')
        _DS1_LOADED = True
        break
if not _DS1_LOADED:
    print('WARNING: DS1 not found — skipped')

for fname, loader in [
    ('causal_dataset_2.json',  _load_ds2),
    ('causal_dataset_3.json',  _load_ds3),
    ('causal_dataset_4.json',  _load_ds4),
    ('causal_dataset_5.json',  _load_ds5),
    ('causal_dataset_6a.json', _load_ds6a),
]:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f'WARNING: {fname} not found — skipped')
        continue
    raw = json.load(open(fpath))
    loaded = loader(raw)
    all_entries.extend(loaded)
    ds = loaded[0]['dataset'] if loaded else fname
    print(f'Loaded {ds}: {len(loaded)} entries')

# ── Summary ───────────────────────────────────────────────────────────────────
rows = [{'id': e['id'], 'dataset': e['dataset'], 'source': e['source'],
         'n_relations': len(e['relations']),
         'has_chain': bool(e.get('chain')),
         'chain_len': len(e['chain']) if e.get('chain') else 0,
         'has_direction': bool(e.get('connector_direction')),
         'has_g4': bool(e.get('expected_g4_mechanism')),
         'challenge_type': e.get('challenge_type', ''),
         'tier': e.get('tier', ''),
         'unresolved_offsets': sum(
             1 for rel in e['relations']
             if rel.get('cause_start', 0) == -1 or rel.get('effect_start', 0) == -1
         )} for e in all_entries]
df_meta = pd.DataFrame(rows)

agg = df_meta.groupby('dataset').agg(
    entries=('id', 'count'),
    avg_rels=('n_relations', 'mean'),
    chain_entries=('has_chain', 'sum'),
    direction_annot=('has_direction', 'sum'),
    g4_annot=('has_g4', 'sum'),
    unresolved_spans=('unresolved_offsets', 'sum'),
).round(1)

print(f'\nTotal entries: {len(all_entries)}  '
      f'({df_meta.dataset.value_counts().sort_index().to_dict()})')
print()
display(agg)
print()
print('Notes:')
print('  ds1: connector_direction + expected_g4_mechanism annotated')
print('  ds2: connector_direction / expected_g4_mechanism not annotated yet')
print('  ds3: annotated offsets re-derived via text.find(span) (Fix 5)')
print('  ds4: 4 annotation word-order mismatches corrected in source file (Fix 5)')
print('  ds5: 30 records, 72 relations, 12 challenge types (incl. 2 cross_paragraph);')
print('       multi-cause records expanded; XPG-* paragraphs normalised via _join_text_list()')
print('  ds6a: 44 records, 97 relations, 8 challenge types, 2 tiers;')
print('        causal_status + attribution fields available per relation')
print('  unresolved_spans=0 means all spans are locatable in their source text')


## Evaluation helpers

In [ ]:
def token_set(text):
    """Lowercase token set, ignoring punctuation."""
    return set(re.findall(r'\b\w+\b', text.lower()))

def token_f1(pred, gold):
    """Token-level F1 between two spans (standard NER span metric)."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    p_toks, g_toks = token_set(pred), token_set(gold)
    tp = len(p_toks & g_toks)
    if tp == 0:
        return 0.0
    prec = tp / len(p_toks)
    rec  = tp / len(g_toks)
    return 2 * prec * rec / (prec + rec)

def jaccard(a, b):
    a, b = token_set(a), token_set(b)
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def best_stmt_match(stmts, gt_cause, gt_effect):
    """Return the extracted statement that best covers (gt_cause, gt_effect)."""
    best, best_score = None, -1.0
    for s in stmts:
        score = (token_f1(s.get('cause_text', ''), gt_cause) +
                 token_f1(s.get('effect_text', ''), gt_effect)) / 2
        if score > best_score:
            best_score = score
            best = s
    return best, best_score

def mock_span(text, score=0.9):
    """Create a minimal ResolvedSpan for entity-linking tests."""
    return ResolvedSpan(
        span_id='mock', doc_id='mock', start=0, end=len(text),
        text=text, labels=['deg_mech'], groups=['G4_MECHANISM_PROCESS'],
        provenance={'sources': [{'score': score, 'source_type': 'gazetteer'}]}
    )

print('Helpers defined.')

---
## Section 1 — Extraction: per-entry results

Run `extract_stage5_causal_condition` on every entry and display a side-by-side comparison of extracted vs ground-truth cause/effect spans.

In [ ]:
results = []

for entry in all_entries:
    stage5 = extract_stage5_causal_condition(
        doc_id=entry['id'],
        chunk_index=0,
        chunk_text=entry['text'],
        doc_type='WO' if entry['source'] == 'work_order' else 'CR',
        section_role='body',
        nlp=nlp,
    )
    stmts  = stage5.get('extracted_causal_statements', [])
    chains = stage5.get('causal_chain', [])

    # For each GT relation, find the best-matching extracted statement
    rel_matches = []
    for rel in entry['relations']:
        best, score = best_stmt_match(stmts, rel['cause_span'], rel['effect_span'])
        rel_matches.append({'rel': rel, 'best_stmt': best, 'score': score})

    results.append({
        'entry': entry,
        'stage5': stage5,
        'stmts': stmts,
        'chains': chains,
        'rel_matches': rel_matches,
    })

# ── Display ───────────────────────────────────────────────────────────────────
BAR  = '─' * 80
BAR2 = '  ' + '·' * 40

for r in results:
    e  = r['entry']
    ds = e['dataset'].upper()
    print(BAR)
    print(f"[{e['id']}] {ds}  |  {len(e['relations'])} GT relation(s)  |  "
          f"extracted stmts={len(r['stmts'])}  chains={len(r['chains'])}")
    # Show only first 120 chars of text to keep output compact
    print(f"TEXT: {e['text'][:120]}{'…' if len(e['text'])>120 else ''}")
    for j, rm in enumerate(r['rel_matches']):
        rel = rm['rel']
        b   = rm['best_stmt']
        print(BAR2)
        print(f"  GT-R{j+1}  cause : {rel['cause_span']}")
        print(f"  GT-R{j+1}  effect: {rel['effect_span']}")
        if b:
            cf1 = token_f1(b.get('cause_text', ''), rel['cause_span'])
            ef1 = token_f1(b.get('effect_text', ''), rel['effect_span'])
            print(f"  EX     cause : {b.get('cause_text','')}  (F1={cf1:.2f})")
            print(f"  EX     effect: {b.get('effect_text','')}  (F1={ef1:.2f})")
            print(f"  source={b.get('source','?')}  conf={b.get('confidence',0):.2f}  avg-F1={rm['score']:.2f}")
        else:
            print('  *** NO STATEMENT EXTRACTED ***')
print(BAR)
print(f'\nTotal: {len(results)} entries processed.')


---
## Section 2 — Span quality (Stage 1: noun-chunk spans)

Token-F1 between the extracted best-match cause/effect and the ground-truth spans.
Threshold: F1 ≥ 0.5 = partial match; F1 = 1.0 = exact token match.

In [ ]:
span_rows = []
for r in results:
    e = r['entry']
    for rm in r['rel_matches']:
        rel = rm['rel']
        b   = rm['best_stmt']
        if b:
            cf1 = token_f1(b.get('cause_text', ''), rel['cause_span'])
            ef1 = token_f1(b.get('effect_text', ''), rel['effect_span'])
        else:
            cf1 = ef1 = 0.0
        span_rows.append({
            'id': e['id'],
            'dataset': e['dataset'],
            'relation_type': rel.get('relation_type', ''),
            'source': b.get('source', 'none') if b else 'none',
            'cause_F1': round(cf1, 3),
            'effect_F1': round(ef1, 3),
            'avg_F1': round((cf1 + ef1) / 2, 3),
            'extracted': bool(b),
        })

df_span = pd.DataFrame(span_rows)
display(df_span.style.background_gradient(
    subset=['cause_F1', 'effect_F1', 'avg_F1'], cmap='RdYlGn'))

print('\n── Overall aggregate ──────────────────────────────────────')
total_rels = len(df_span)
print(f"  Total GT relations   : {total_rels}")
print(f"  Extraction rate      : {df_span.extracted.mean():.0%}  "
      f"({df_span.extracted.sum()}/{total_rels})")
print(f"  Mean cause  F1       : {df_span.cause_F1.mean():.3f}")
print(f"  Mean effect F1       : {df_span.effect_F1.mean():.3f}")
print(f"  Mean avg    F1       : {df_span.avg_F1.mean():.3f}")
print(f"  Exact cause  (F1=1.0): {(df_span.cause_F1==1.0).sum()}")
print(f"  Exact effect (F1=1.0): {(df_span.effect_F1==1.0).sum()}")
print(f"  Partial match (F1>=0.5): {(df_span.avg_F1>=0.5).sum()}")
print(f"  No match (F1=0.0)    : {(df_span.avg_F1==0.0).sum()}")

print('\n── By dataset ─────────────────────────────────────────────')
display(df_span.groupby('dataset')[['cause_F1', 'effect_F1', 'avg_F1', 'extracted']].agg(
    ['mean', 'count']).round(3))

print('\n── By extractor source ────────────────────────────────────')
display(df_span.groupby('source')[['cause_F1', 'effect_F1', 'avg_F1']].agg(
    ['mean', 'count']).round(3))

# J2 — Precision metric
# For each extracted statement, check if it matches ANY gold relation in its entry.
# This measures how many extractions are useful, regardless of which GT relation
# they align with (important for DS3/DS6a with multiple relations per entry).
print('\n── J2: Precision (extracted stmts matching any GT relation) ──')
prec_rows = []
for r in results:
    e = r['entry']
    for s in r['stmts']:
        matched_any = any(
            best_stmt_match([s], rel['cause_span'], rel['effect_span'])[1] >= 0.5
            for rel in e['relations']
        )
        prec_rows.append({
            'dataset': e['dataset'],
            'source':  s.get('source', 'unknown'),
            'matched': matched_any,
        })

if prec_rows:
    df_prec = pd.DataFrame(prec_rows)
    total_extracted = len(df_prec)
    overall_prec = df_prec.matched.mean()
    print(f"  Total extracted stmts: {total_extracted}")
    print(f"  Overall precision    : {overall_prec:.1%}  "
          f"({df_prec.matched.sum()}/{total_extracted})")
    print()
    display(df_prec.groupby('dataset').agg(
        n_extracted=('matched', 'count'),
        precision=('matched', 'mean'),
    ).round(3))
else:
    print('  No extracted statements to evaluate.')


---
## Section 3 — Connector direction accuracy

For each entry the dataset records `connector_direction` (forward / backward / null).
We check whether the extracted statement's cause/effect assignment agrees with ground truth
by comparing which ground-truth span the extracted `cause_text` overlaps more.

In [ ]:
# Direction test is DS1-only (connector_direction only annotated there)
dir_rows = []
for r in results:
    e = r['entry']
    if e['dataset'] != 'ds1':
        continue
    gt_dir = e.get('connector_direction')
    # Use the first (and only) rel_match for DS1
    rm = r['rel_matches'][0] if r['rel_matches'] else None
    b  = rm['best_stmt'] if rm else None
    if gt_dir is None or b is None:
        continue

    ex_cause  = b.get('cause_text', '')
    ex_effect = b.get('effect_text', '')
    gt_cause  = e['relations'][0]['cause_span']
    gt_effect = e['relations'][0]['effect_span']

    cause_to_gt_cause  = jaccard(ex_cause, gt_cause)
    cause_to_gt_effect = jaccard(ex_cause, gt_effect)

    if cause_to_gt_cause + cause_to_gt_effect == 0:
        predicted_dir = 'unknown'
    elif cause_to_gt_cause >= cause_to_gt_effect:
        predicted_dir = 'forward'
    else:
        predicted_dir = 'backward'

    correct = predicted_dir == gt_dir
    dir_rows.append({
        'id': e['id'],
        'connector': e['relations'][0]['connective'],
        'gt_direction': gt_dir,
        'predicted_direction': predicted_dir,
        'correct': correct,
        'cause↔gt_cause_J': round(cause_to_gt_cause, 3),
        'cause↔gt_effect_J': round(cause_to_gt_effect, 3),
    })

df_dir = pd.DataFrame(dir_rows)
display(df_dir.style.apply(
    lambda col: ['background-color: #c8e6c9' if v else 'background-color: #ffcdd2'
                 for v in col] if col.name == 'correct' else ['']*len(col), axis=0
))

n_correct = df_dir.correct.sum()
n_total   = len(df_dir)
print(f'\nDirection accuracy (DS1): {n_correct}/{n_total} = {n_correct/n_total:.0%}')
inc = df_dir[~df_dir.correct][["id", "connector", "gt_direction", "predicted_direction"]]
if len(inc):
    print('Incorrect:')
    print(inc.to_string(index=False))
else:
    print('All correct!')


---
## Section 4 — Chain detection (Stage 2)

For the 7 multi-hop entries, compare `causal_chain` nodes against the expected `chain` list.

**Metrics:**
- *Chain found* — did the extractor produce at least one chain with length ≥ 2?
- *Node Jaccard* — best-match Jaccard between any predicted chain node and each expected node
- *Full-chain score* — mean node Jaccard across all expected nodes (0 = no reconstruction, 1 = perfect)

In [ ]:
chain_rows = []
for r in results:
    e = r['entry']
    if not e.get('chain'):
        continue

    gt_chain    = e['chain']
    pred_chains = r['chains']
    chain_found = bool(pred_chains)

    # Best predicted chain = longest (already sorted)
    if pred_chains:
        best_pred_nodes = pred_chains[0]['nodes']
    else:
        # Fall back: cause+effect from all extracted statements
        best_pred_nodes = []
        for s in r['stmts']:
            c = (s.get('cause_text') or '').strip()
            ef = (s.get('effect_text') or '').strip()
            if c and c not in best_pred_nodes:
                best_pred_nodes.append(c)
            if ef and ef not in best_pred_nodes:
                best_pred_nodes.append(ef)

    # Per-node F1 against best matching predicted node
    node_scores = [
        max((token_f1(pn, gn) for pn in best_pred_nodes), default=0.0)
        for gn in gt_chain
    ]
    full_chain_score = float(np.mean(node_scores)) if node_scores else 0.0

    chain_rows.append({
        'id': e['id'],
        'dataset': e['dataset'],
        'expected_nodes': ' → '.join(gt_chain),
        'predicted_nodes': ' → '.join(best_pred_nodes) if best_pred_nodes else '—',
        'chain_found': chain_found,
        'node_F1_per_node': [round(s, 3) for s in node_scores],
        'full_chain_score': round(full_chain_score, 3),
    })

df_chain = pd.DataFrame(chain_rows)

# Verbose display
for _, row in df_chain.iterrows():
    found_tag = '✓' if row['chain_found'] else '✗'
    print(f"[{row['id']}] {row['dataset'].upper()}  chain_found={found_tag}  "
          f"full_chain_score={row['full_chain_score']}")
    print(f"  GT : {row['expected_nodes'][:100]}")
    print(f"  EX : {row['predicted_nodes'][:100]}")
    print(f"  node F1s: {row['node_F1_per_node']}")
    print()

print('── Aggregate ──────────────────────────────────────────────')
print(f"  Total chain entries  : {len(df_chain)}")
print(f"  Chain detection rate : {df_chain.chain_found.mean():.0%}  "
      f"({df_chain.chain_found.sum()}/{len(df_chain)})")
print(f"  Mean full-chain score: {df_chain.full_chain_score.mean():.3f}")
print()
print('── By dataset ─────────────────────────────────────────────')
display(df_chain.groupby('dataset').agg(
    entries=('id', 'count'),
    chain_found=('chain_found', 'mean'),
    mean_chain_score=('full_chain_score', 'mean'),
).round(3))


---
## Section 5 — Entity linking (Stage 3)

Test `_best_overlapping_entity` directly against the 10 entries with a known `expected_g4_mechanism`.
We create a mock `ResolvedSpan` from the expected mechanism label and check which pass (substring, Jaccard, lemma) fires first.

In [ ]:
# Entity linking is DS1-only (expected_g4_mechanism only annotated there)
el_rows = []
for r in results:
    e = r['entry']
    mech = e.get('expected_g4_mechanism')
    if mech is None:
        continue

    cause_text = e['relations'][0]['cause_span']
    mock_spans = [mock_span(mech)]

    # Test each pass independently
    hit_p1 = _best_overlapping_entity(cause_text, mock_spans, nlp=None) is not None

    def _jaccard_check(target, span_text):
        toks = lambda t: set(w for w in t.lower().split() if len(w) > 1)
        a, b = toks(target), toks(span_text)
        if not a or not b: return 0.0
        return len(a & b) / len(a | b)
    p2_score = _jaccard_check(cause_text, mech)
    hit_p2 = p2_score >= _ENTITY_LINK_JACCARD_THRESHOLD

    def _lemma_check(target, span_text):
        def lems(t):
            return set(tok.lemma_.lower() for tok in nlp(t)
                       if not tok.is_punct and not tok.is_space)
        tl, sl = lems(target), lems(span_text)
        tstr, sstr = ' '.join(sorted(tl)), ' '.join(sorted(sl))
        if sstr and (sstr in tstr or tstr in sstr):
            return True
        if not tl or not sl: return False
        return len(tl & sl) / len(tl | sl) >= _ENTITY_LINK_JACCARD_THRESHOLD
    hit_p3 = (not hit_p1 and not hit_p2) and _lemma_check(cause_text, mech)

    full_result = _best_overlapping_entity(cause_text, mock_spans, nlp=nlp)
    hit_full = full_result is not None

    pass_fired = ('pass1' if hit_p1 else
                  'pass2' if hit_p2 else
                  'pass3' if hit_p3 else
                  'none')

    el_rows.append({
        'id': e['id'],
        'cause_text': cause_text,
        'expected_g4': mech,
        'hit': hit_full,
        'pass_fired': pass_fired,
        'p2_jaccard': round(p2_score, 3),
    })

df_el = pd.DataFrame(el_rows)
display(df_el.style.apply(
    lambda col: ['background-color: #c8e6c9' if v else 'background-color: #ffcdd2'
                 for v in col] if col.name == 'hit' else ['']*len(col), axis=0
))

print(f'\nEntity linking hit-rate (DS1): {df_el.hit.mean():.0%}  '
      f'({df_el.hit.sum()}/{len(df_el)})')
print('Hits by pass:')
print(df_el.groupby('pass_fired').size().to_string())


---
## Section 6 — Threshold calibration

Sweep `_CHAIN_JACCARD_THRESHOLD` and `_ENTITY_LINK_JACCARD_THRESHOLD` over the dataset
to find values that maximise performance. Recommended operating points are marked.

In [ ]:
import matplotlib.pyplot as plt

# ── Chain Jaccard threshold sweep (all datasets with chain annotations) ───────
thresholds = np.arange(0.10, 0.85, 0.05)
chain_scores_by_thresh = []

def _tok(text):
    return frozenset(t.lower() for t in (text or '').split() if len(t) > 1)

def _jac(a, b):
    if not a or not b: return 0.0
    return len(a & b) / len(a | b)

for thresh in thresholds:
    scores = []
    for r in results:
        e = r['entry']
        if not e.get('chain'):
            continue
        stmts = r['stmts']
        n = len(stmts)
        if n == 0:
            scores.append(0.0)
            continue

        succ = {i: [] for i in range(n)}
        pred = {i: [] for i in range(n)}
        for i in range(n):
            eff = _tok(stmts[i].get('effect_text') or '')
            for j in range(n):
                if i == j: continue
                cau = _tok(stmts[j].get('cause_text') or '')
                if _jac(eff, cau) >= thresh:
                    succ[i].append(j)
                    pred[j].append(i)

        found_chains = []
        sources = [i for i in range(n) if not pred[i]] or list(range(n))

        def dfs(path, visited, _succ=succ, _fc=found_chains):
            cur = path[-1]
            children = [j for j in _succ[cur] if j not in visited]
            if not children:
                if len(path) >= 2:
                    _fc.append(path)
                return
            for c in children:
                dfs(path + [c], visited | {c}, _succ, _fc)

        for src in sources:
            dfs([src], {src})

        gt_chain = e['chain']
        if found_chains:
            best_nodes = []
            for si in found_chains[0]:
                if not best_nodes:
                    c = (stmts[si].get('cause_text') or '').strip()
                    if c:
                        best_nodes.append(c)
                ef = (stmts[si].get('effect_text') or '').strip()
                if ef:
                    best_nodes.append(ef)
        else:
            best_nodes = []

        node_f1s = [max((token_f1(pn, gn) for pn in best_nodes), default=0.0)
                    for gn in gt_chain]
        scores.append(float(np.mean(node_f1s)) if node_f1s else 0.0)

    chain_scores_by_thresh.append(float(np.mean(scores)) if scores else 0.0)

best_chain_idx   = int(np.argmax(chain_scores_by_thresh))
best_chain_thresh = thresholds[best_chain_idx]
n_chain_entries  = sum(1 for r in results if r['entry'].get('chain'))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, chain_scores_by_thresh, 'o-', color='steelblue',
        label=f'mean full-chain score (n={n_chain_entries})')
ax.axvline(best_chain_thresh, color='crimson', linestyle='--',
           label=f'best T={best_chain_thresh:.2f}')
ax.axvline(_CHAIN_JACCARD_THRESHOLD, color='orange', linestyle=':',
           label=f'current T={_CHAIN_JACCARD_THRESHOLD}')
ax.set_xlabel('_CHAIN_JACCARD_THRESHOLD')
ax.set_ylabel('Mean full-chain node F1')
ax.set_title('Stage 2 — Chain Jaccard threshold calibration (all datasets)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Chain entries used    : {n_chain_entries}')
print(f'Current threshold     : {_CHAIN_JACCARD_THRESHOLD}')
print(f'Best threshold        : {best_chain_thresh:.2f}  '
      f'(score={chain_scores_by_thresh[best_chain_idx]:.3f})')


In [ ]:
# ── Entity-linking Jaccard threshold sweep (DS1 only) ────────────────────────
el_scores_by_thresh = []

for thresh in thresholds:
    hits = []
    for r in results:
        e = r['entry']
        mech = e.get('expected_g4_mechanism')
        if mech is None:
            continue
        cause_text = e['relations'][0]['cause_span']

        # Pass 1: substring
        tl = cause_text.lower(); sl = mech.lower()
        if sl and (sl in tl or tl in sl):
            hits.append(1); continue

        # Pass 2: Jaccard at this threshold
        toks = lambda t: set(w for w in t.lower().split() if len(w) > 1)
        a, b_set = toks(cause_text), toks(mech)
        jac = len(a & b_set) / len(a | b_set) if (a and b_set) else 0.0
        if jac >= thresh:
            hits.append(1); continue

        # Pass 3: lemma
        lems = lambda t: set(tok.lemma_.lower() for tok in nlp(t)
                             if not tok.is_punct and not tok.is_space)
        tl2, sl2 = lems(cause_text), lems(mech)
        tstr, sstr = ' '.join(sorted(tl2)), ' '.join(sorted(sl2))
        if sstr and (sstr in tstr or tstr in sstr):
            hits.append(1); continue
        lem_jac = len(tl2 & sl2) / len(tl2 | sl2) if (tl2 and sl2) else 0.0
        hits.append(1 if lem_jac >= thresh else 0)

    el_scores_by_thresh.append(float(np.mean(hits)) if hits else 0.0)

best_el_idx   = int(np.argmax(el_scores_by_thresh))
best_el_thresh = thresholds[best_el_idx]
n_el_entries  = sum(1 for r in results if r['entry'].get('expected_g4_mechanism'))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, el_scores_by_thresh, 'o-', color='darkorange',
        label=f'hit-rate (n={n_el_entries})')
ax.axvline(best_el_thresh, color='crimson', linestyle='--',
           label=f'best T={best_el_thresh:.2f}')
ax.axvline(_ENTITY_LINK_JACCARD_THRESHOLD, color='orange', linestyle=':',
           label=f'current T={_ENTITY_LINK_JACCARD_THRESHOLD}')
ax.set_xlabel('_ENTITY_LINK_JACCARD_THRESHOLD')
ax.set_ylabel('Entity linking hit-rate')
ax.set_title('Stage 3 — Entity-linking Jaccard threshold calibration (DS1)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Current threshold : {_ENTITY_LINK_JACCARD_THRESHOLD}')
print(f'Best threshold    : {best_el_thresh:.2f}  '
      f'(hit-rate={el_scores_by_thresh[best_el_idx]:.0%})')


---
## Section 7 — Summary dashboard

In [ ]:
import warnings
warnings.filterwarnings('ignore')

total_rels    = len(df_span)
n_extracted   = df_span.extracted.sum()
mean_cause_f1 = df_span.cause_F1.mean()
mean_eff_f1   = df_span.effect_F1.mean()
mean_avg_f1   = df_span.avg_F1.mean()
dir_acc       = df_dir.correct.mean() if len(df_dir) else float('nan')
chain_rate    = df_chain.chain_found.mean() if len(df_chain) else float('nan')
chain_score   = df_chain.full_chain_score.mean() if len(df_chain) else float('nan')
el_hitrate    = df_el.hit.mean() if len(df_el) else float('nan')
precision     = df_prec.matched.mean() if 'df_prec' in dir() and len(df_prec) else float('nan')

ds_counts = {ds: df_meta[df_meta.dataset == ds].shape[0]
             for ds in ['ds1', 'ds2', 'ds3', 'ds4', 'ds5', 'ds6a']}

summary = pd.DataFrame([
    {'Stage': '—', 'Metric': 'Total entries',
     'Value': (f'{len(all_entries)}  ' +
               '  '.join(f'{ds}={ds_counts.get(ds,0)}'
                         for ds in ['ds1','ds2','ds3','ds4','ds5','ds6a']
                         if ds_counts.get(ds, 0) > 0))},
    {'Stage': '—', 'Metric': 'Total GT relations',
     'Value': f'{total_rels}'},
    {'Stage': '—', 'Metric': 'Extraction rate (≥1 stmt)',
     'Value': f'{n_extracted/total_rels:.0%} ({n_extracted}/{total_rels})'},
    {'Stage': 'Stage 1', 'Metric': 'Mean cause span F1',
     'Value': f'{mean_cause_f1:.3f}'},
    {'Stage': 'Stage 1', 'Metric': 'Mean effect span F1',
     'Value': f'{mean_eff_f1:.3f}'},
    {'Stage': 'Stage 1', 'Metric': 'Mean avg span F1',
     'Value': f'{mean_avg_f1:.3f}'},
    {'Stage': 'J2',      'Metric': 'Precision (stmts matching any GT relation)',
     'Value': (f'{precision:.1%}  ({int(df_prec.matched.sum())}/{len(df_prec)})'
               if not np.isnan(precision) else 'n/a')},
    {'Stage': 'fix',     'Metric': 'Connector direction accuracy (DS1)',
     'Value': f'{dir_acc:.0%}  ({df_dir.correct.sum()}/{len(df_dir)})' if len(df_dir) else 'n/a'},
    {'Stage': 'Stage 2', 'Metric': 'Chain detection rate',
     'Value': (f'{chain_rate:.0%}  ({int(df_chain.chain_found.sum())}/{len(df_chain)})'
               if len(df_chain) else 'n/a')},
    {'Stage': 'Stage 2', 'Metric': 'Mean full-chain node F1',
     'Value': f'{chain_score:.3f}' if not np.isnan(chain_score) else 'n/a'},
    {'Stage': 'Stage 2', 'Metric': 'Calibrated chain threshold',
     'Value': f'{best_chain_thresh:.2f}  (current: {_CHAIN_JACCARD_THRESHOLD})'},
    {'Stage': 'Stage 3', 'Metric': 'Entity linking hit-rate (DS1)',
     'Value': (f'{el_hitrate:.0%}  ({int(df_el.hit.sum())}/{len(df_el)})'
               if len(df_el) else 'n/a')},
    {'Stage': 'Stage 3', 'Metric': 'Calibrated entity-link threshold',
     'Value': f'{best_el_thresh:.2f}  (current: {_ENTITY_LINK_JACCARD_THRESHOLD})'},
])

display(summary.style.set_properties(**{'text-align': 'left'}))

print('\n── Per-dataset avg span F1 ────────────────────────────────')
display(df_span.groupby('dataset')[['cause_F1', 'effect_F1', 'avg_F1']].mean().round(3))

print('\n── Extractor source breakdown ─────────────────────────────')
display(df_span.groupby(['dataset','source']).size().rename('count').to_frame())

# 14.2 — FM-resolution metric (primary KPI for Improvements G–J)
# Measures whether the extracted cause_text resolves to the correct inferred_fm_label
# via _best_overlapping_entity. More aligned with RCA workflow value than token F1.
print('\n── 14.2: FM-resolution rate (primary KPI) ─────────────────')
print('   Does extracted cause_text link to the expected G4 mechanism via entity linker?')
fm_rows = []
for r in results:
    e = r['entry']
    mech = e.get('expected_g4_mechanism')
    if mech is None:
        continue
    for s in r['stmts']:
        cause_text = s.get('cause_text', '')
        linked = _best_overlapping_entity(
            cause_text,
            [mock_span(mech)],
            nlp=nlp,
        )
        fm_rows.append({
            'id':           e['id'],
            'dataset':      e['dataset'],
            'cause_text':   cause_text,
            'expected_fm':  mech,
            'fm_resolved':  linked is not None,
            'source':       s.get('source', ''),
        })

if fm_rows:
    df_fm = pd.DataFrame(fm_rows)
    print(f"  Entries with FM annotation: {df_meta['has_g4'].sum()}")
    print(f"  Extracted stmts evaluated : {len(df_fm)}")
    print(f"  FM-resolution rate        : {df_fm.fm_resolved.mean():.0%}  "
          f"({df_fm.fm_resolved.sum()}/{len(df_fm)})")
    print()
    display(df_fm.groupby('dataset').agg(
        n_stmts=('fm_resolved', 'count'),
        fm_resolution_rate=('fm_resolved', 'mean'),
    ).round(3))
    print()
    print('  Unresolved cases (cause_text did not link to expected FM):')
    unresolved_fm = df_fm[~df_fm.fm_resolved][['id', 'cause_text', 'expected_fm']]
    if len(unresolved_fm):
        print(unresolved_fm.to_string(index=False))
    else:
        print('  None — all extracted causes resolved to their expected FM.')
else:
    print('  No entries with expected_g4_mechanism annotation found.')
    print('  Annotate DS1 entries with expected_g4_mechanism to enable this metric.')

# DS5 / DS6a challenge-type breakdown
for ds_name, ds_label in [('ds5', 'DS5'), ('ds6a', 'DS6a')]:
    if ds_name not in df_span['dataset'].values:
        continue
    print(f'\n── {ds_label}: span F1 by challenge type ──────────────────')
    ds_entries = {e['id']: e for e in all_entries if e['dataset'] == ds_name}
    df_ds = df_span[df_span.dataset == ds_name].copy()
    df_ds['challenge_type'] = df_ds['id'].map(
        lambda eid: ds_entries.get(eid, {}).get('challenge_type', '')
    )
    display(df_ds.groupby('challenge_type').agg(
        n_relations=('avg_F1', 'count'),
        extracted=('extracted', 'mean'),
        cause_F1=('cause_F1', 'mean'),
        effect_F1=('effect_F1', 'mean'),
        avg_F1=('avg_F1', 'mean'),
    ).round(3))

# DS6a tier breakdown
if 'ds6a' in df_span['dataset'].values:
    print('\n── DS6a: span F1 by tier ───────────────────────────────────')
    df_6a = df_span[df_span.dataset == 'ds6a'].copy()
    tier_map = {e['id']: e.get('tier', '') for e in all_entries if e['dataset'] == 'ds6a'}
    df_6a['tier'] = df_6a['id'].map(tier_map)
    display(df_6a.groupby('tier').agg(
        n_relations=('avg_F1', 'count'),
        extracted=('extracted', 'mean'),
        avg_F1=('avg_F1', 'mean'),
    ).round(3))
